# 05 — Backbone lab (experimental)

> **Experimental / internal — not part of the public product promise.**

Notebook **04** shows **admin-lane V0 bundles**. This notebook shows **lab consumer formats** (`consumer-catalog`, `frontend-contract`, `ai-context`) gated by RBAC policy.

Requires `clearmetric-core[runtime]` for DuckDB query cells.

In [ ]:
import importlib.util
import os
import sys
import urllib.error
import urllib.request
from pathlib import Path
from types import ModuleType

_COLD_START_GITHUB_RAW_BASE = (
    "https://raw.githubusercontent.com/ClearMetric-Labs/ClearMetric-Core/main"
)

_CACHED_NOTEBOOKS_DIR = Path(
    "/Users/jonkim/.cache/clearmetric/github-main/examples/notebooks"
)


def _github_raw_base() -> str:
    paths = sys.modules.get("_paths")
    default = (
        paths.GITHUB_RAW_BASE if paths is not None else _COLD_START_GITHUB_RAW_BASE
    )
    return os.environ.get("CM_CLEARMETRIC_GITHUB_RAW_BASE", default)


def _fetch_repo_file(repo_relative: str, dest: Path) -> None:
    if dest.is_file():
        return
    paths = sys.modules.get("_paths")
    if paths is not None:
        paths._fetch_github_file(repo_relative, dest)
        return
    url = f"{_github_raw_base()}/{repo_relative}"
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=60) as response:
            dest.write_bytes(response.read())
    except urllib.error.URLError as exc:
        raise FileNotFoundError(
            f"Failed to download {url}. Check network access and branch path."
        ) from exc


def _find_local_helpers(start: Path | None = None) -> Path | None:
    start = start or Path.cwd()
    for root in (start, *start.parents):
        nested = root / "examples" / "notebooks"
        if (nested / "_paths.py").is_file():
            return nested
        if (root / "_paths.py").is_file() and (root / "_notebook_setup.py").is_file():
            return root
    return None


def _resolve_setup_file(start: Path | None = None) -> Path:
    local = _find_local_helpers(start)
    if local is not None:
        return local / "_notebook_setup.py"
    paths = sys.modules.get("_paths")
    cache_dir = (
        paths.CACHED_NOTEBOOKS_DIR if paths is not None else _CACHED_NOTEBOOKS_DIR
    )
    setup_file = cache_dir / "_notebook_setup.py"
    if not setup_file.is_file():
        _fetch_repo_file("examples/notebooks/_notebook_setup.py", setup_file)
    return setup_file


def _exec_setup_module(setup_file: Path) -> ModuleType:
    spec = importlib.util.spec_from_file_location("_notebook_setup", setup_file)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load {setup_file}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


_exec_setup_module(_resolve_setup_file()).setup_notebook(extras="runtime")

In [ ]:
import json

from _paths import backbone_lab_project

os.environ["CM_EXPERIMENTAL"] = "1"
PROJECT_DIR = backbone_lab_project()
IDENTITY = "analyst"
print(f"project: {PROJECT_DIR}")

## Discover and ingest

In [ ]:
from clearmetric.adapters import ingest_source
from clearmetric.compiler import compile, discover
from clearmetric.core import load_project_config

project = load_project_config(PROJECT_DIR)
for src in discover(PROJECT_DIR).sources:
    print(f"{src.kind}\t{src.path}")
for kind in ("warehouse", "dbt", "intent"):
    artifact = ingest_source(kind, project)
    print(f"{kind}: nodes={len(artifact.nodes)} edges={len(artifact.edges)}")

compiled = compile(PROJECT_DIR)

## Admin vs consumer catalog

In [ ]:
from clearmetric.emitters import emit_compile
from clearmetric.emitters.registry import LAB_COMPILE_FORMATS

admin_catalog = json.loads(emit_compile("catalog", compiled))
consumer = json.loads(emit_compile("consumer-catalog", compiled, identity=IDENTITY))
admin_ids = {n["id"] for n in admin_catalog["nodes"]}
consumer_ids = {n["id"] for n in consumer["payload"]["nodes"]}
print("lab formats:", LAB_COMPILE_FORMATS)
print(f"admin nodes: {len(admin_ids)}  consumer nodes: {len(consumer_ids)}")
print("consumer-only examples:", sorted(consumer_ids - admin_ids))

## Policy-gated impact

`orders.amount --upstream` is empty in this lab because the fixture has no column-level derivation into `orders.amount`. To demonstrate gated impact, trace the query's declared dependency instead: the analyst can see the related column, while an unprivileged viewer is denied at the selection.

In [ ]:
from clearmetric.compiler.impact import impact
from clearmetric.core.errors import PolicyDeniedError

SELECTION = "query:executive_revenue"
DIRECTION = "downstream"
_, ungated = impact(PROJECT_DIR, selection=SELECTION, direction=DIRECTION)
_, analyst = impact(
    PROJECT_DIR,
    selection=SELECTION,
    direction=DIRECTION,
    identity=IDENTITY,
)
assert analyst.related_ids == ["column:orders.amount"]
print(f"selection: {analyst.selection_id}")
print("ungated related ids:", ungated.related_ids)
print("analyst related ids:", analyst.related_ids)

try:
    impact(PROJECT_DIR, selection=SELECTION, direction=DIRECTION, identity="viewer")
except PolicyDeniedError as exc:
    print("viewer denied:", exc)

## Frontend contract and AI context

In [ ]:
contracts = json.loads(emit_compile("frontend-contract", compiled, identity=IDENTITY))
ai_context = json.loads(emit_compile("ai-context", compiled, identity=IDENTITY))
print("consumer envelope keys:", sorted(contracts.keys()))
print("frontend-contract query:", contracts["payload"]["queries"][0]["id"])
print(
    "ai-context node kinds:",
    sorted({node["id"].split(":")[0] for node in ai_context["payload"]["nodes"]}),
)

## Execute query (requires runtime)

In [ ]:
from clearmetric.runtime import execute_project_query

rows = execute_project_query(
    compiled.artifact,
    identity=IDENTITY,
    rules_path=compiled.project.policy.rules,
    query_selection="query:executive_revenue",
    project_dir=PROJECT_DIR,
)
print(rows)